# 02 — EDA and Preprocessing

This notebook continues from Phase 1. It reuses the target and leakage exclusions stored in the Phase 1 summary, then demonstrates cleaning, feature encoding, model-specific scaling, outlier reporting, and exploratory plots. No predictive model is trained.

## Load the data and fixed Phase 1 decisions

Reading the decisions from the saved report prevents this phase from silently choosing a different target or leakage policy.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_prep import (
    build_model_preprocessors, clean_dataset, identify_feature_roles,
    load_dataset, load_phase1_decisions
)
from src.eda import (
    class_balance_summary, iqr_outlier_report, plot_behaviour_vs_target,
    plot_correlation_heatmap, plot_target_distribution, print_class_balance,
    print_encoding_decisions, print_missing_decisions, render_phase2_summary,
    select_behaviour_columns, select_key_numeric_columns
)

sns.set_theme(style='whitegrid', context='notebook')
df, csv_path = load_dataset(PROJECT_ROOT / 'data')
target, leakage = load_phase1_decisions(PROJECT_ROOT / 'results' / 'dataset_summary.txt')
print('Loaded:', csv_path.resolve())
print('Fixed target from Phase 1:', target)
print('Fixed leakage exclusions from Phase 1:', leakage)

Loaded: P:\College\Sem VII Acad\IPRM\Project\data\Ecommerce.csv
Fixed target from Phase 1: purchased
Fixed leakage exclusions from Phase 1: ['revenue', 'cart_abandoned', 'rating', 'review_text', 'review_helpful_votes', 'payment_method', 'revenue_normalized']


## Missing values and duplicate rows

The reusable policy retains complete columns, drops rows with missing labels, drops features above 40% missingness, median-imputes other numeric fields, and mode-imputes other categorical fields. Every actual decision is printed. Exact duplicates are removed before imputation and checked again afterward.

In [2]:
cleaned, missing_decisions, duplicate_report = clean_dataset(df, target)
print_missing_decisions(missing_decisions)
print('\nDuplicate handling:', duplicate_report)
print('Remaining missing values:', int(cleaned.isna().sum().sum()))
print('Cleaned shape:', cleaned.shape)

Missing-value decisions by column:
- customer_id [numeric]: retain (no imputation needed) - 0.00% missing
- session_id [numeric]: retain (no imputation needed) - 0.00% missing
- visit_date [categorical/text]: retain (no imputation needed) - 0.00% missing
- device_type [numeric]: retain (no imputation needed) - 0.00% missing
- user_type [numeric]: retain (no imputation needed) - 0.00% missing
- marketing_channel [numeric]: retain (no imputation needed) - 0.00% missing
- product_id [numeric]: retain (no imputation needed) - 0.00% missing
- product_category [numeric]: retain (no imputation needed) - 0.00% missing
- unit_price [numeric]: retain (no imputation needed) - 0.00% missing
- quantity [numeric]: retain (no imputation needed) - 0.00% missing
- discount_percent [numeric]: retain (no imputation needed) - 0.00% missing
- discount_amount [numeric]: retain (no imputation needed) - 0.00% missing
- revenue [numeric]: retain (no imputation needed) - 0.00% missing
- pages_viewed [numeric]: 

## Feature roles and categorical encoding

Nominal labels and integer codes without meaningful order use one-hot encoding only when cardinality is at most 20. A high-cardinality non-identifier category uses frequency encoding, while high-cardinality raw identifiers are dropped. The discovered duration bucket uses ordinal encoding because its labels have a natural sequence. The redundant raw date string and Phase 1 leakage fields are not model features.

In [3]:
roles = identify_feature_roles(cleaned, target, leakage)
print_encoding_decisions(roles)

print('\nNumeric fields:', roles['numeric'])
print('Nominal one-hot fields:', roles['nominal'])
print('Frequency-encoded fields:', roles['frequency'])
print('Ordinal fields:', roles['ordinal'])
print('Excluded fields:', roles['excluded'])

Feature-role and encoding decisions:
- customer_id: excluded identifier; none - identifier-like field with 8442 unique values (33.77% of rows) could encourage memorisation
- session_id: excluded identifier; none - identifier-like field with 25000 unique values (100.00% of rows) could encourage memorisation
- visit_date: excluded raw date; none - date-like text would create high-cardinality dummy variables; calendar fields are retained separately
- device_type: nominal categorical; one-hot - 3 observed labels/codes have no defensible numeric order
- user_type: nominal categorical; one-hot - 2 observed labels/codes have no defensible numeric order
- marketing_channel: nominal categorical; one-hot - 6 observed labels/codes have no defensible numeric order
- product_id: excluded identifier; none - identifier-like field with 899 unique values (3.60% of rows) could encourage memorisation
- product_category: nominal categorical; one-hot - 8 observed labels/codes have no defensible numeric ord

## Model-specific preprocessing views

Both views apply the same one-hot and ordinal encoding. Tree models retain original numeric scales; logistic regression receives standardized numeric values. These full-data transformations only verify the Phase 2 implementation and dimensions. In Phase 3, fresh preprocessors must be fitted inside training folds.

In [4]:
X = cleaned.drop(columns=[target])
y = cleaned[target].copy()
tree_demo, linear_demo = build_model_preprocessors(roles)
X_tree = tree_demo.fit_transform(X)
X_linear = linear_demo.fit_transform(X)
encoded_shapes = {'tree': X_tree.shape, 'linear': X_linear.shape}
print('Tree feature matrix shape:', X_tree.shape, '- numeric values are not scaled')
print('Logistic feature matrix shape:', X_linear.shape, '- numeric values are standardized')

scaler = linear_demo.named_transformers_['numeric'].named_steps['scaler']
scaling_details = pd.DataFrame({
    'numeric_column': roles['numeric'],
    'training_mean_used_in_demo': scaler.mean_,
    'training_scale_used_in_demo': scaler.scale_
})
print('\nObserved scaling parameters for the demonstration view:')
print(scaling_details.to_string(index=False))

Tree feature matrix shape: (25000, 52) - numeric values are not scaled
Logistic feature matrix shape: (25000, 52) - numeric values are standardized

Observed scaling parameters for the demonstration view:
  numeric_column  training_mean_used_in_demo  training_scale_used_in_demo
      unit_price                  782.319010                   476.602636
        quantity                    2.489040                     1.114540
discount_percent                    8.998800                     9.263455
 discount_amount                  174.997669                   269.000031
    pages_viewed                   12.535840                     6.929623
time_on_site_sec                  903.262920                   518.665828
   added_to_cart                    0.644680                     0.478610
       visit_day                   15.714520                     8.796162


## IQR outlier check

The 1.5×IQR rule is applied to discovered quantitative behavior fields. Counts are reported, but no rows are silently deleted because unusual values may still be valid customer behavior.

In [5]:
key_numeric = select_key_numeric_columns(cleaned, roles)
outliers = iqr_outlier_report(cleaned, key_numeric)
print('Key numeric columns checked:', key_numeric)
print(outliers.to_string(index=False))
print('\nRows removed due to outliers: 0')

Key numeric columns checked: ['pages_viewed', 'time_on_site_sec', 'quantity', 'discount_amount', 'discount_percent', 'unit_price']
          column       q1        q3     iqr  lower_bound  upper_bound  outlier_count  outlier_percent                                               decision
    pages_viewed   7.0000   19.0000  12.000     -11.0000      37.0000              0            0.000 retain; report only pending domain/model justification
time_on_site_sec 453.0000 1355.0000 902.000    -900.0000    2708.0000              0            0.000 retain; report only pending domain/model justification
        quantity   1.0000    3.0000   2.000      -2.0000       6.0000              0            0.000 retain; report only pending domain/model justification
 discount_amount   0.0000  240.3550 240.355    -360.5325     600.8875           1883            7.532 retain; report only pending domain/model justification
discount_percent   0.0000   15.0000  15.000     -22.5000      37.5000              0

## Target distribution and class balance

Class proportions guide the split strategy and evaluation metrics used later. The plot is saved to `results/figures/` and displayed from that saved file.

In [6]:
figure_dir = PROJECT_ROOT / 'results' / 'figures'
balance = class_balance_summary(cleaned, target)
print_class_balance(balance)
target_plot = plot_target_distribution(cleaned, target, figure_dir)
image = plt.imread(target_plot)
plt.figure(figsize=(9, 6))
plt.imshow(image)
plt.axis('off')
plt.show()

Target class balance:
- class 0: 19384 rows (77.54%)
- class 1: 5616 rows (22.46%)
The target is moderately imbalanced (majority:minority ratio 3.45:1). Later evaluation should use stratified splits and metrics beyond accuracy.


C:\Users\Parth\AppData\Local\Temp\ipykernel_27448\3943045136.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Correlation heatmap

The heatmap uses only retained quantitative fields plus the target. Persisted leakage fields and identifier codes are excluded so the display does not present misleading post-outcome relationships.

In [7]:
correlation_plot = plot_correlation_heatmap(cleaned, target, roles, figure_dir)
image = plt.imread(correlation_plot)
plt.figure(figsize=(11, 9))
plt.imshow(image)
plt.axis('off')
plt.show()

C:\Users\Parth\AppData\Local\Temp\ipykernel_27448\1645603181.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Behavioral fields versus the target

The plotted columns are selected from actual retained fields using behavior-related name concepts. Low-cardinality fields show observed target rates; continuous fields use box plots.

In [8]:
behaviour_columns = select_behaviour_columns(cleaned, target, roles)
behaviour_plots = plot_behaviour_vs_target(cleaned, target, behaviour_columns, roles, figure_dir)
print('Actual behavior columns selected:', behaviour_columns)
for path in behaviour_plots:
    image = plt.imread(path)
    plt.figure(figsize=(9, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

Actual behavior columns selected: ['added_to_cart', 'pages_viewed', 'session_duration_bucket', 'time_on_site_sec']


C:\Users\Parth\AppData\Local\Temp\ipykernel_27448\4205250465.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save auditable Phase 2 outputs

Decision and outlier tables are saved as CSV files, and the complete observed summary is saved as text. This keeps the notebook, modules, and generated results consistent.

In [9]:
tables_dir = PROJECT_ROOT / 'results' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(missing_decisions).to_csv(tables_dir / 'missing_value_decisions.csv', index=False)
pd.DataFrame(roles['decisions']).to_csv(tables_dir / 'encoding_decisions.csv', index=False)
outliers.to_csv(tables_dir / 'outlier_report.csv', index=False)

figure_paths = [target_plot, correlation_plot, *behaviour_plots]
phase2_summary = render_phase2_summary(
    csv_path, target, leakage, duplicate_report, missing_decisions, roles,
    encoded_shapes, outliers, balance, behaviour_columns, figure_paths
)
summary_path = PROJECT_ROOT / 'results' / 'eda_preprocessing_summary.txt'
summary_path.write_text(phase2_summary, encoding='utf-8')
print('Saved Phase 2 summary:', summary_path.resolve())
print('Saved figures:', [path.name for path in figure_paths])
print('No predictive model was trained in this phase.')

Saved Phase 2 summary: P:\College\Sem VII Acad\IPRM\Project\results\eda_preprocessing_summary.txt
Saved figures: ['target_class_distribution.png', 'correlation_heatmap.png', 'behaviour_added_to_cart_vs_target.png', 'behaviour_pages_viewed_vs_target.png', 'behaviour_session_duration_bucket_vs_target.png', 'behaviour_time_on_site_sec_vs_target.png']
No predictive model was trained in this phase.


## Phase 2 boundary

Cleaning, preprocessing design, outlier reporting, and EDA are complete. Train/test splitting, model fitting, tuning, and evaluation are intentionally deferred to Phase 3.